In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import os
from PIL import Image
import torchvision
import matplotlib.pyplot as plt

In [ ]:
class ImageProcessor:
    def __init__(self, root, transformations = None):
        self.root = root
        self.transformations = transformations

        # List of Paths of all images...
        self.all_image_paths = [os.path.join(root, img) for img in os.listdir(root)]

    def __len__(self):
        return len(self.all_image_paths)

    def __getitem__(self, idx):
        img_path = self.all_image_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [ ]:
root = "./Images"
transformations = transforms.Compose([
    transforms.CenterCrop(178),    # Cropping ==> (178x218) to (178x178)
    transforms.Resize(64),         # Resize ==> (178x178) to (64x64)
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # [-1, 1]
])

In [ ]:
dataset = ImageProcessor(root, transformations)

In [ ]:
print(len(dataset))

In [ ]:
Dataloader = DataLoader(dataset, batch_size = 128, shuffle=True)

## Generator Network

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3):
        super(Generator, self).__init__()

        # Fully Connected Layers...
        self.model = nn.Sequential(

            nn.Linear(z_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),
            
            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64 * 64 * img_channels),
            nn.Tanh()     # [-1, 1]
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img

        # Fake image will be returned => 64 x 64 x 3 x batch_size

## Discriminator Network

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels = 3): # 3 is for RGB
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),

            nn.Linear(img_channels * 64 * 64, 1024),
            nn.LeakyReLU(0.2, inplace = True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace = True), 

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace = True),

            nn.Linear(256, 1)  
        )
        
    def forward(self, img):
        return self.model(img)

In [ ]:
device = torch.device("cuda")

In [ ]:
GAN_loss = nn.BCEWithLogitsLoss()

generator = Generator().to(device)
g_optimizer = optim.Adam(generator.parameters(), lr = 0.0002, betas = (0.5, 0.999))

discriminator = Discriminator().to(device)
d_optimizer = optim.Adam(discriminator.parameters(), lr = 0.0002, betas = (0.5, 0.999))

## Training

In [ ]:
checkpoint = torch.load(
    "vanilla_gan_checkpoint.pth",
    map_location=device,
    weights_only=True
)

generator.load_state_dict(checkpoint['generator_state_dict'])
discriminator.load_state_dict(checkpoint['discriminator_state_dict'])

g_optimizer.load_state_dict(checkpoint['g_optimizer_state_dict'])
d_optimizer.load_state_dict(checkpoint['d_optimizer_state_dict'])

In [ ]:
def generate_new(i):
    generated_imgs = generator(torch.randn(10, 100).to(device))
    generated_imgs = generated_imgs.detach().cpu()
    
    grid = torchvision.utils.make_grid(generated_imgs, nrow = 5, normalize = True) 
    # Normalise is to convert images from [-1, 1] to [0, 1]

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.title(f"Batches Done: {i + 1}")
    plt.axis("off")
    plt.show()

In [ ]:
for i, img in enumerate(Dataloader):
    real_imgs = img.to(device)
    batch_size = real_imgs.size(0)

    # Creating Labels (y-values)
    labels_for_Real = torch.ones(batch_size, 1).to(device) # [1, 1, 1....]
    labels_for_Fake = torch.zeros(batch_size, 1).to(device) # [0, 0, 0....]

    # Producing Fake Images
    Fake_Images = generator(torch.randn(batch_size, 100).to(device))

    
    # Training the Dicriminator
    d_optimizer.zero_grad()
    
    # Losses for Discriminator
    Real_images_loss = GAN_loss(discriminator(real_imgs), labels_for_Real)    # (predicted, labels)
    Fake_images_loss = GAN_loss(discriminator(Fake_Images.detach()), labels_for_Fake)    # (predicted, labels)

    d_loss = (Real_images_loss + Fake_images_loss) / 2
    
    d_loss.backward()
    d_optimizer.step()


    # Training the Generator
    g_optimizer.zero_grad()

    # Losses for Generator
    # This one is really really nice
    g_loss = GAN_loss(discriminator(Fake_Images), labels_for_Real)    # (predicted, labels)
    
    g_loss.backward()
    g_optimizer.step()
    
    if (i + 1) % 100 == 0: 
        print(f"Batch: {i + 1} --- G-loss: {g_loss} --- D-loss: {d_loss}")
        # Generating 10 new Images after every 10 Batches Epoch
        generate_new(i)

In [ ]:
# generate(-1)

In [ ]:
checkpoint = {
    'generator_state_dict': generator.state_dict(),
    'discriminator_state_dict': discriminator.state_dict(),
    'g_optimizer_state_dict': g_optimizer.state_dict(),
    'd_optimizer_state_dict': d_optimizer.state_dict()
    }
    
torch.save(checkpoint, "vanilla_gan_checkpoint.pth")
print(f"New Checkpoint Saved!")